# RAG (Retrieval-Augmented Generation)

RAG combines **information retrieval + language generation**.

Instead of asking an LLM to answer only from its internal knowledge:

**User Query → Retrieve relevant documents → Give retrieved context to LLM → Generate answer**

## Basic Workflow

**Load Documents → Clean → Chunk → Embed → Store in Vector Database → Query → Retrieve → Build Context → Generate Answer → Evaluate → Improve**

## RAG Architecture

```text
Documents
   ↓
Load
   ↓
Clean
   ↓
Chunk
   ↓
Embedding Model
   ↓
Vector Database
   ↓
User Query
   ↓
Query Embedding
   ↓
Similarity Search
   ↓
Top-K Relevant Chunks
   ↓
Prompt + Context
   ↓
LLM
   ↓
Answer
```

### Main components

- **Document Loader** → reads PDFs, web pages, DOCX, TXT, etc.
- **Text Splitter** → breaks documents into chunks
- **Embedding Model** → converts text into vectors
- **Vector Database** → stores/searches vectors
- **Retriever** → finds relevant chunks
- **LLM** → generates the final answer
- **Prompt** → tells the LLM how to use retrieved context

## Libraries / Tools

Common RAG ecosystem:

- **LangChain** → orchestration/components
- **LlamaIndex** → data/RAG workflows
- **FAISS** → local vector similarity search
- **Chroma** → vector database
- **Pinecone** → managed vector database
- **Weaviate / Qdrant / Milvus** → vector databases
- **Sentence Transformers** → embeddings
- **Hugging Face Transformers** → embedding/LLM ecosystem
- **OpenAI / other LLM APIs** → generation

Use one stack for a project. Do not collect vector databases like Pokémon.

Install common libraries

In [ ]:
# Example packages
# pip install langchain langchain-community langchain-text-splitters
# pip install sentence-transformers faiss-cpu
# pip install pypdf
# pip install transformers

# Step 1: Load Documents

**Use:** Read source documents that will become the knowledge base.

In [ ]:
# Example: PDF loading with LangChain

# from langchain_community.document_loaders import PyPDFLoader
#
# loader = PyPDFLoader("document.pdf")
# documents = loader.load()
#
# print("Documents/pages:", len(documents))
# print(documents[0].page_content[:500])

# Step 2: Text Cleaning

**Use:** Remove unnecessary noise before creating chunks.

Typical actions:
- Remove unwanted whitespace
- Remove repeated headers/footers when appropriate
- Preserve useful headings
- Preserve metadata such as page number/source

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Example:
# for doc in documents:
#     doc.page_content = clean_text(doc.page_content)

# Step 3: Chunking

**Use:** Split long documents into smaller pieces that can be retrieved effectively.

Important parameters:
- `chunk_size`
- `chunk_overlap`

Bad chunking can hurt retrieval even when the embedding model is good.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=800,
#     chunk_overlap=100
# )
#
# chunks = text_splitter.split_documents(documents)
#
# print("Number of chunks:", len(chunks))
# print(chunks[0].page_content[:500])

# Step 4: Embeddings

**Use:** Convert each text chunk into a numerical vector so semantically similar text can be searched.

```text
Text chunk
   ↓
Embedding model
   ↓
Vector
```

Examples:
- Sentence Transformers
- OpenAI embedding models
- Hugging Face embedding models

In [ ]:
# Example with Sentence Transformers

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

# Example:
# texts = [chunk.page_content for chunk in chunks]
# vectors = embedding_model.encode(texts)
#
# print(vectors.shape)

# Step 5: Vector Database

**Use:** Store embeddings and retrieve similar chunks efficiently.

For a local/simple project, FAISS is a common option.

In [ ]:
import faiss

# Example:
# dimension = vectors.shape[1]
#
# index = faiss.IndexFlatL2(dimension)
# index.add(vectors.astype("float32"))
#
# print("Stored vectors:", index.ntotal)

# Step 6: Retrieval

**Use:** Convert the user query into an embedding and retrieve the most relevant chunks.

Main parameter:
- `top_k` → number of chunks retrieved

In [ ]:
# Example similarity search

# query = "What is the refund policy?"
#
# query_vector = embedding_model.encode(
#     [query]
# ).astype("float32")
#
# distances, indices = index.search(
#     query_vector,
#     k=5
# )
#
# retrieved_chunks = [
#     chunks[i]
#     for i in indices[0]
# ]
#
# for chunk in retrieved_chunks:
#     print(chunk.page_content[:300])
#     print("---")

# Step 7: Prompt + Context

**Use:** Give the retrieved information to the LLM and instruct it to answer from that context.

Basic pattern:

```text
System instruction
+
Retrieved context
+
User question
→
LLM
→
Answer
```

A good prompt should tell the model what to do when the answer is not present in the retrieved context.

In [ ]:
# Example prompt template

# context = "\n\n".join(
#     chunk.page_content
#     for chunk in retrieved_chunks
# )
#
# prompt = f'''
# Answer the question using only the context below.
# If the answer is not in the context, say that
# the information is not available.
#
# Context:
# {context}
#
# Question:
# {query}
# '''

# Step 8: Generation

**Use:** Send the query + retrieved context to an LLM and generate the final answer.

The LLM is the **generator**, not the knowledge-base search engine.

In [ ]:
# Conceptual LLM call

# response = llm.invoke(prompt)
# print(response)

# Complete RAG Pipeline

```text
Documents
   ↓
Loader
   ↓
Cleaning
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Store
   ↓
Retriever
   ↓
Relevant Context
   ↓
Prompt
   ↓
LLM
   ↓
Answer
```

## RAG Types

### Naive / Basic RAG
Simple:
**chunk → embed → retrieve → prompt → answer**

### Advanced RAG
Adds techniques such as:
- Better chunking
- Metadata filtering
- Query rewriting
- Hybrid search
- Reranking
- Context compression
- Multiple retrieval strategies

### Hybrid Search
Combines:
- Keyword search
- Vector/semantic search

Useful when exact terms and semantic meaning both matter.

### Reranking
Retrieve a larger candidate set first, then use a reranker to select the most relevant chunks.

## RAG Evaluation

RAG has **two separate parts to evaluate**.

### Retrieval Evaluation

Ask:

**Did we retrieve the right information?**

Metrics/concepts:
- Recall@K
- Precision@K
- Hit Rate
- MRR
- NDCG

### Generation Evaluation

Ask:

**Did the LLM produce a good answer from the retrieved context?**

Important concepts:
- Faithfulness / groundedness
- Answer relevance
- Context relevance
- Correctness

A RAG system can fail even when the LLM is excellent if retrieval gives it the wrong context.

In [ ]:
# Simple retrieval evaluation concept

# If relevant_document_ids are known:
#
# retrieved_ids = [...]
# relevant_ids = set([...])
#
# hit = any(
#     doc_id in relevant_ids
#     for doc_id in retrieved_ids
# )
#
# print("Hit:", hit)

# Model / RAG Comparison

Compare the whole pipeline, not only the LLM.

Possible experiments:

**Embedding Model A**
vs.
**Embedding Model B**

**Chunk size 500**
vs.
**Chunk size 1000**

**Vector Search**
vs.
**Hybrid Search**

**Top-K = 3**
vs.
**Top-K = 5**

**With Reranker**
vs.
**Without Reranker**

Then evaluate:
- Retrieval quality
- Answer quality
- Latency
- Cost

# Model Improvement

**Baseline RAG → Better Documents → Better Chunking → Better Embeddings → Better Retrieval → Reranking → Better Prompt → Evaluate → Final RAG**

### 1. Improve Data
- Better document extraction
- Remove irrelevant content
- Preserve metadata
- Deduplicate documents

### 2. Improve Chunking
Tune:
- `chunk_size`
- `chunk_overlap`
- Semantic/structure-aware splitting

### 3. Improve Retrieval
Try:
- Higher/lower `top_k`
- Metadata filtering
- Hybrid search
- Query rewriting
- Multi-query retrieval

### 4. Add Reranking
Retrieve more candidates → rerank → send only the best context to the LLM.

### 5. Improve Prompt
Tell the LLM:
- Use retrieved context
- Do not invent unsupported facts
- Cite/source retrieved information when required
- Say when information is unavailable

### 6. Improve Embeddings
Compare embedding models using a retrieval validation set.

### 7. Evaluate End-to-End
Measure both:
**Retrieval quality + Generation quality**

# RAG vs Fine-Tuning

### RAG
Use when:
- Knowledge changes frequently
- You need answers from private documents
- You want source-grounded answers
- You need to update knowledge without retraining the model

### Fine-Tuning
Use when:
- You want to change model behavior/style
- You need task-specific behavior
- You have suitable training examples

RAG primarily changes **what information the model sees**.

Fine-tuning primarily changes **how the model behaves**.

# Quick Revision

**RAG** → retrieve information before generating an answer

**Chunking** → split documents into useful retrieval units

**Embedding** → convert text into vectors

**Vector Database** → store/search vectors

**Retriever** → find relevant chunks

**Reranker** → reorder retrieved candidates by relevance

**Prompt** → combine question + context + instructions

**LLM** → generate answer

### Main workflow

**Documents → Clean → Chunk → Embed → Store → Retrieve → Rerank → Prompt → LLM → Answer → Evaluate → Improve**

### Main failure points

**Bad documents → Bad chunks → Bad retrieval → Bad context → Bad answer**

A fancy LLM cannot magically retrieve information that you failed to retrieve.